# Flower Species Classification Using Convolutional Neural Networks

**SPEC 2111 – Deep Learning / Neural Networks Project**

**Team Members:** Bharathi Mamillapalli (D24130833), Aisling OBroin (D24130826)

---

## 1. Problem Definition & Objectives

### 1.1 Problem Statement
Automatic flower species identification is a challenging computer vision task with applications in biodiversity monitoring, plant identification apps, and botanical research.

### 1.2 Motivation
Manual flower identification requires botanical expertise. CNNs can automate this process effectively.

### 1.3 Project Objectives
1. Classify 102 flower species from Oxford Flowers 102 dataset
2. Compare architectures: ResNet, VGG, EfficientNet (with transfer learning)under class-imbalanced conditions
    - VGG-16: a classic archictecture (linear stack of layers)
    - ResNet-18: introduces "Skip Connections" to help gradients flow in deeper networks
    - EfficientNet-B0: uses "Compound Scaling" to balance depth, width, and resolution efficiently
3. Target: **mean per-class accuracy > 80%**
4. Metrics: mean per-class accuracy, overall accuracy, macro F1-score

## 2. Data Collection & Exploratory Data Analysis (EDA)

*Approach aligned with lecturer's notebooks (Week 5–6, Improving Performance): ReLU, Softmax, Cross-Entropy, Adam, Dropout, Early Stopping, L2 regularization.*

### 2.1 Dataset Description
- **Dataset:** Oxford Flowers 102
- **Source:** [Kaggle - The Oxford Flowers 102 Dataset](https://www.kaggle.com/datasets/waseemalastal/the-oxford-flowers-102-dataset)
- **Size:** 8,189 images across 102 flower classes
- **Splits:** Train (6,552) / Validation (819) / Test (819)
- **Class balance:** 40–258 images per class

In [ ]:
import sys
# install to the current Jupyter kernel
!{sys.executable} -m pip install scipy
import scipy
print(f"Scipy version {scipy.__version__} is successfully installed!")

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.3.2 -> 26.0.1
[notice] To update, run: /data/.shared-kernel-envs/perftools-py3-11/.venv/bin/python -m pip install --upgrade pip
Scipy version 1.11.4 is successfully installed!


In [ ]:
import sys
# This installs PyTorch and Torchvision directly into your Jupyter kernel
!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
^C
ERROR: Operation cancelled by user


In [ ]:
# Run this cell FIRST - enables plot display in Jupyter
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except:
    import matplotlib
    matplotlib.use('Agg')

In [ ]:
# Install required packages (run once if using Colab or fresh environment)
# !pip install torch torchvision pandas matplotlib seaborn scikit-learn tqdm

%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Verify setup - run this cell first. If you see "Setup OK" below, imports and matplotlib work.
plt.rcParams['figure.figsize'] = (10, 6)
fig, ax = plt.subplots()
ax.plot([1, 2, 3], [1, 4, 2])
ax.set_title("Setup test - if you see this plot, matplotlib is working")
plt.show()
print("Setup OK - you should see a small plot above.")

### 2.2 Data Loading

We use **torchvision.datasets.Flowers102** which downloads the dataset automatically from the official Oxford source. Requires `scipy` for loading `.mat` files. Alternative: [Kaggle](https://www.kaggle.com/datasets/waseemalastal/the-oxford-flowers-102-dataset).

In [ ]:
# Use torchvision's built-in Flowers102 - fully reproducible, auto-downloads
# Dataset: https://www.robots.ox.ac.uk/~vgg/data/flowers/102/
# Alternative Kaggle: https://www.kaggle.com/datasets/waseemalastal/the-oxford-flowers-102-dataset

DATA_ROOT = "./data"  # Dataset will be downloaded here if not present

# Transforms: ImageNet normalization for pretrained models
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets (download=True fetches from Oxford if not present)
train_ds = datasets.Flowers102(root=DATA_ROOT, split="train", transform=transform_train, download=True)
val_ds = datasets.Flowers102(root=DATA_ROOT, split="val", transform=transform_eval, download=True)
test_ds = datasets.Flowers102(root=DATA_ROOT, split="test", transform=transform_eval, download=True)

BATCH_SIZE = 32
NUM_WORKERS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Class names (Flowers102 uses numeric labels 0-101)
NUM_CLASSES = 102
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}, Classes: {NUM_CLASSES}")

### 2.3 Exploratory Data Analysis (EDA)

We visualize the dataset structure: sample images, class distribution, and image characteristics.

In [ ]:
# Sample images from different classes
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
axes = axes.flatten()
# Get one image per class for first 15 classes
shown = set()
idx = 0
for i in range(len(train_ds)):
    img, label = train_ds[i]
    if label not in shown and len(shown) < 15:
        ax = axes[len(shown)]
        # Denormalize for display
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"Class {label}", fontsize=9)
        ax.axis("off")
        shown.add(label)
    if len(shown) >= 15:
        break
plt.suptitle("Sample Flowers from Oxford 102 Dataset", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution (images per class in train set)
from collections import Counter
train_labels = [train_ds[i][1] for i in range(len(train_ds))]
counts = Counter(train_labels)
classes = sorted(counts.keys())
values = [counts[c] for c in classes]

plt.figure(figsize=(14, 4))
plt.bar(classes, values, color="steelblue", edgecolor="navy", alpha=0.8)
plt.xlabel("Class ID")
plt.ylabel("Number of Images")
plt.title("Class Distribution in Training Set (40-258 images per class)")
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("Dataset Summary:")
print(f"  Train: {len(train_ds)} images")
print(f"  Val:   {len(val_ds)} images")
print(f"  Test:  {len(test_ds)} images")
print(f"  Min images/class: {min(values)}, Max: {max(values)}")

In [ ]:
# Calculate weights: 1 / frequency
class_counts = np.array(values)
weights = 1.0 / class_counts
weights = torch.FloatTensor(weights).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

## 3. Model Design & Implementation

We use **transfer learning** with pretrained ImageNet models. The final fully-connected layer is replaced to output 102 classes. We compare:
- **ResNet18** – Residual connections, 18 layers
- **VGG16** – Classic deep CNN
- **EfficientNet-B0** – Efficient architecture with compound scaling

**Following lecturer's approach (Week 6 / Improving Performance):**
- **ReLU** hidden layers (avoids vanishing gradients; lecturer recommends over sigmoid)
- **Softmax** for multi-class output
- **Cross-Entropy** loss (best for classification)
- **Adam** optimizer (lecturer: "Try Adam first")
- **Dropout** for overfitting prevention
- **L2 regularization** via weight_decay (penalize squared weights)

In [ ]:
def get_model(arch="resnet18", num_classes=102, pretrained=True, dropout_rate=0.5):
    """Create model with pretrained backbone, Dropout (lecturer's overfitting prevention), and classifier head."""
    if arch == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(model.fc.in_features, num_classes)
        )
    elif arch == "vgg16":
        model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        model.classifier[6] = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(4096, num_classes)
        )
    elif arch == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        in_f = model.classifier[1].in_features
        model.classifier[1] = nn.Sequential(nn.Dropout(dropout_rate), nn.Linear(in_f, num_classes))
    else:
        raise ValueError(f"Unknown architecture: {arch}")
    return model.to(device)

## 4. Training & Evaluation

### 4.1 Training Loop
We train using Weighted Cross-Entropy Loss to account for class imbalance (40–258 images per class), ensuring that minority classes contribute equally to the gradient updates. The models are optimized using the Adam optimizer with L2 regularization (weight decay), and we track training loss alongside validation metrics (Accuracy, MPCA, and Macro F1) to generate performance curves.

In [ ]:
def mean_per_class_accuracy(y_true, y_pred, num_classes=102):
    """Compute mean per-class accuracy (primary metric for Oxford 102)."""
    accs = []
    for c in range(num_classes):
        mask = y_true == c
        if mask.sum() > 0:
            accs.append((y_pred[mask] == c).mean().item())
    return np.mean(accs) * 100 if accs else 0.0

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(loader, desc="Train", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), total, correct

def evaluate(model, loader, criterion, num_classes=102):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            loss = criterion(out, labels)
            total_loss += loss.item()
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = accuracy_score(all_labels, all_preds) * 100
    mean_acc = mean_per_class_accuracy(all_labels, all_preds, num_classes)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0) * 100
    return total_loss / len(loader), acc, mean_acc, f1

In [ ]:
def train_model(arch, train_loader, val_loader, test_loader, weights=None, epochs=10, lr=1e-3, weight_decay=1e-4, patience=5):
    """Train with Adam, L2 regularization (weight_decay),and Class Weights for imbalance."""
    model = get_model(arch, NUM_CLASSES)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)  # L2 via weight_decay

    history = {"loss": [], "val_loss": [], "val_acc": [], "val_mean_acc": [], "val_f1": []}
    best_mean_acc = 0
    best_state = None
    epochs_without_improvement = 0

    for ep in range(epochs):
        t_loss, _, _ = train_epoch(model, train_loader, criterion, optimizer)
        v_loss, v_acc, v_mean_acc, v_f1 = evaluate(model, val_loader, criterion, NUM_CLASSES)
        history["loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["val_acc"].append(v_acc)
        history["val_mean_acc"].append(v_mean_acc)
        history["val_f1"].append(v_f1)
        print(f"Epoch {ep+1}/{epochs} | Loss: {t_loss:.4f} | Val Acc: {v_acc:.2f}% | Mean Per-Class: {v_mean_acc:.2f}% | F1: {v_f1:.2f}%")
        if v_mean_acc > best_mean_acc:
            best_mean_acc = v_mean_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {ep+1} (no improvement for {patience} epochs)")
            break

    # Load best model and evaluate on test
    if best_state is not None:
        model.load_state_dict(best_state)
    model = model.to(device)
    test_loss, test_acc, test_mean_acc, test_f1 = evaluate(model, test_loader, criterion, NUM_CLASSES)
    return history, {"acc": test_acc, "mean_acc": test_mean_acc, "f1": test_f1}, model

## 5. Experiments & Alternatives

We compare three architectures and optionally vary hyperparameters (learning rate, epochs). For a full submission, increase `epochs` (e.g. 20–30) for better convergence.

In [ ]:
# Experiment 1: ResNet18
QUICK_TEST = True   # Set to False for full submission (10+ epochs per model)
EPOCHS = 2 if QUICK_TEST else 10   # 2 = quick test (~5 min/model); 10+ for real results
print("=== Training ResNet18 ===")
print(f"Epochs: {EPOCHS} (QUICK_TEST={QUICK_TEST})")
hist_resnet, res_resnet, model_resnet = train_model("resnet18", train_loader, val_loader, test_loader, weights=weights,epochs=EPOCHS)

In [ ]:
# Experiment 2: VGG16
print("=== Training VGG16 ===")
hist_vgg, res_vgg, model_vgg = train_model("vgg16", train_loader, val_loader, test_loader, weights=weights, epochs=EPOCHS)

In [ ]:
# Experiment 3: EfficientNet-B0
print("=== Training EfficientNet-B0 ===")
hist_eff, res_eff, model_eff = train_model("efficientnet_b0", train_loader, val_loader, test_loader, weights=weights, epochs=EPOCHS)

### 5.2 Alternative: Learning Rate Comparison (Optional)

Comparing different learning rates for the best architecture.

In [ ]:
# Optional: Compare learning rates (e.g. 1e-3 vs 1e-4) - uncomment to run
# lr_results = {}
# for lr in [1e-3, 1e-4]:
#     _, res, _ = train_model("resnet18", train_loader, val_loader, test_loader, epochs=5, lr=lr)
#     lr_results[lr] = res["mean_acc"]
# print("LR comparison:", lr_results)

### 5.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in [("ResNet18", hist_resnet), ("VGG16", hist_vgg), ("EfficientNet-B0", hist_eff)]:
    axes[0].plot(hist["loss"], label=name)
    axes[1].plot(hist["val_mean_acc"], label=name)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Mean Per-Class Accuracy (%)")
axes[1].set_title("Validation Mean Per-Class Accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to training_curves.png")

## 6. Results & Discussion

### 6.1 Model Comparison

In [ ]:
# Summary table
results_df = pd.DataFrame({
    "Model": ["ResNet18", "VGG16", "EfficientNet-B0"],
    "Overall Acc (%)": [res_resnet["acc"], res_vgg["acc"], res_eff["acc"]],
    "Mean Per-Class Acc (%)": [res_resnet["mean_acc"], res_vgg["mean_acc"], res_eff["mean_acc"]],
    "Macro F1 (%)": [res_resnet["f1"], res_vgg["f1"], res_eff["f1"]]
})
print(results_df.to_string(index=False))
print(f"\nTarget: Mean per-class accuracy > 80%")
best_model = max(["ResNet18", "VGG16", "EfficientNet-B0"],
                 key=lambda m: res_resnet if m=="ResNet18" else (res_vgg if m=="VGG16" else res_eff)["mean_acc"])
best_res = res_eff if best_model=="EfficientNet-B0" else (res_vgg if best_model=="VGG16" else res_resnet)
print(f"Best: {best_model} with mean per-class accuracy {best_res['mean_acc']:.2f}%")

In [ ]:
# 1. Identify the best model based on your target metric
best_name = results_df.loc[results_df["Mean Per-Class Acc (%)"].idxmax(), "Model"]
model_map = {"ResNet18": model_resnet, "VGG16": model_vgg, "EfficientNet-B0": model_eff}
best_model_obj = model_map[best_name]

# 2. Generate predictions on the Test Set
all_preds, all_labels = [], []
best_model_obj.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = best_model_obj(imgs)
        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())

# 3. Create the Matrix
cm = confusion_matrix(all_labels, all_preds)

# 4. Plotting with specific adjustments for 102 classes
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues", annot=False, cbar=True) # annot=False because numbers won't fit
plt.title(f"Confusion Matrix: {best_name}\n(Targeting Mean Per-Class Accuracy)")
plt.xlabel("Predicted Flower Class ID")
plt.ylabel("True Flower Class ID")
plt.tight_layout()

# Save
plt.savefig("confusion_matrix_final.png", dpi=300)
plt.show()

### 6.2 Discussion

**Interpretation of results:**
- **Mean per-class accuracy** is the primary metric for Oxford Flowers 102, as it treats each class equally and is robust to class imbalance (40–258 images per class).
- **Overall accuracy** can be inflated by well-represented classes.
- **Macro F1-score** balances precision and recall across classes.

**Why certain models may perform better:**
- **EfficientNet** typically achieves strong results with fewer parameters due to compound scaling.The use of  "Compound Scaling" to balance depth, width, and resolution, gives the model efficiencies in terms of accuracy vs. parameters
- **ResNet** benefits from residual connections that ease gradient flow in deeper networks.its "Skip Connections" help mitigate the vanishing gradient problem, allowing for more stable training compared to VGG
- **VGG** is deeper but has more parameters and may overfit with limited data. Performance might be hindered by its sheer parameter size (approx. 138M), making it prone to overfitting on a smaller dataset like Oxford 102

**Limitations:**
- Training for only 10 epochs may underutilize the models; 20–30 epochs often improve results.
- Class imbalance may affect minority classes; techniques like oversampling or class weights could help.

###  One question we considered is whether the model struggles more with minority classes (those with only 40 images) even with class weights applied, or are the errors purely based on visual similarity?


## 7. Conclusion & Future Work

### Conclusions
- Transfer learning with pretrained CNNs (ResNet, VGG, EfficientNet) is effective for flower species classification on Oxford Flowers 102.
- The best-performing model (among those compared) achieves competitive mean per-class accuracy toward the 80% target.
- Data augmentation (flip, rotation, color jitter) and ImageNet normalization improve generalization.

### Future Work
1. **Longer training** – Increase epochs to 20–30 for better convergence.
2. **Hyperparameter tuning** – Grid search over learning rate, batch size, and optimizer.
3. **Class balancing** – Apply oversampling or focal loss for minority classes.
4. **Larger models** – Try EfficientNet-B2/B4 or ResNet50 for potential gains.
5. **Ensemble methods** – Combine predictions from multiple models.

In [ ]:
# Final summary - this should always print when you reach the end
print("\n" + "="*50)
print("PROJECT RUN COMPLETE")
print("="*50)
print("Results above: model comparison table, training curves, confusion matrix.")
print("Set QUICK_TEST=False and EPOCHS=10+ for full submission.")